In [53]:
import pandas as pd
import xarray as xr
import numpy as np

In [54]:
def load_grid_info(basin_csv, area_csv):
    country_map = pd.read_csv(basin_csv)  # 必须包含 I,J,country
    area_df     = pd.read_csv(area_csv)     # 必须包含 I,J,以及面积列 Value

    grid_info = country_map.merge(area_df, on=["I", "J"], how="left")
    grid_info.rename(columns={"Value": "cell_area"}, inplace=True)

    return grid_info

In [55]:
def build_basin_matrices(grid_info, lat_size, lon_size):
    basin_matrix = np.empty((lat_size, lon_size), dtype=object)
    frac_matrix  = np.zeros((lat_size, lon_size)) * np.nan
    area_matrix  = np.zeros((lat_size, lon_size)) * np.nan

    for _, r in grid_info.iterrows():
        I = int(r["I"]) - 1
        J = int(r["J"]) - 1

        basin_matrix[I,J] = r["basin"]
        frac_matrix[I,J]  = r["frac"]       # basin 在该格子的比例
        area_matrix[I,J]  = r["cell_area"]  # 格子面积

    return basin_matrix, frac_matrix, area_matrix


In [56]:
def compute_variable_area(ds, varname, year_list, ds_years,
                          basin_matrix, basin_frac_matrix, area_matrix):

    print(f"→ 处理变量 {varname}")
    da = ds[varname].transpose("time","lat","lon")

    unique_basins = pd.unique(basin_matrix.ravel())
    unique_basins = [b for b in unique_basins if b is not None and not pd.isna(b)]

    area_matrix = np.nan_to_num(area_matrix, nan=0.0)
    basin_frac_matrix = np.nan_to_num(basin_frac_matrix, nan=0.0)

    results = []

    for year in year_list:
        t = np.where(ds_years == year)[0][0]

        frac_nc = da[t].values
        frac_nc = np.nan_to_num(frac_nc, nan=0.0)

        # 新公式：加入 basin_frac_matrix
        real_area = frac_nc * area_matrix * basin_frac_matrix

        for basin in unique_basins:
            mask = (basin_matrix == basin)
            total = real_area[mask].sum()

            results.append([varname, basin, year, total])

    return results


In [57]:
def run_area_summary(name_list, year_list, PATH_BASIN, PATH_AREA, PATH_NC, OUTPUT_CSV):

    print("将处理的年份 =", year_list)
    print("将处理的变量 =", name_list)

    # --- 加载 grid 信息 ---
    grid_info = load_grid_info(PATH_BASIN, PATH_AREA)

    # --- 打开 NC 文件 ---
    ds = xr.open_dataset(PATH_NC)
    ds_years = pd.to_datetime(ds["time"].values).year

    lat_size = len(ds["lat"])
    lon_size = len(ds["lon"])

    # --- 构建矩阵 ---
    basin_matrix, basin_frac_matrix, area_matrix = build_basin_matrices(
        grid_info, lat_size, lon_size
    )


    # --- 循环变量 ---
    all_results = []

    for base_name in name_list:
        var_basin  = f"basin_{base_name}"
        var_region = f"region_{base_name}"

        for varname in [var_basin, var_region]:

            res = compute_variable_area(
                ds, varname, year_list, ds_years,
                basin_matrix, basin_frac_matrix, area_matrix
            )
            all_results.extend(res)

    # --- 输出 CSV ---
    df = pd.DataFrame(all_results, columns=["variable", "basin", "year", "area"])
    df.to_csv(OUTPUT_CSV, index=False)

    print(f"完成：结果已保存到 {OUTPUT_CSV}")

    return df


In [ ]:
name_list = ["forest", "agri", "grassland"]    

year_list = list(range(2010, 2101, 10))
year_list.insert(0, 2005)

PATH_COUNTRY = "../../CSV/grid_country_output.csv"
PATH_BASIN = "../../CSV/grid_basin_output.csv"
PATH_AREA    = "../../CSV/GAIJ.csv"
PATH_NC      = "../../NC/compare.nc"
OUTPUT_CSV   = "../../CSV/country_area_timeseries.csv"
output_csv   = "../../CSV/basinarea.csv"

df = run_area_summary(name_list, year_list, PATH_BASIN, PATH_AREA, PATH_NC, output_csv)

将处理的年份 = [2005, 2010, 2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100]
将处理的变量 = ['forest', 'agri', 'grassland']
→ 处理变量 basin_forest
→ 处理变量 region_forest
→ 处理变量 basin_agri
→ 处理变量 region_agri
→ 处理变量 basin_grassland
→ 处理变量 region_grassland
完成：结果已保存到 ../../CSV/country_area_timeseries.csv
